In [ ]:
import warnings
warnings.filterwarnings('ignore')

import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import MultipleLocator, FixedLocator
import seaborn as sns
import plotly.express as px
from scipy.stats import norm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import itertools


from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D 
from sklearn.impute import SimpleImputer

import sys
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
import netket as nk

import json

import time

from flax import nnx
import jax.numpy as jnp
import jax
import math 
import sys
import numpy as np

In [ ]:
def upper(line):
    return line.upper()

In [ ]:
def downsample(iters, values, stride=5):
    return iters[::stride], values[::stride]

In [ ]:
markers = ['o', 's', 'D', '^', 'v', '<', '>', 
           'p', '*', 'h', 'H', '+', 'x', '|', 
           '_', '.', ',', '1', '2', '3', '4']

In [ ]:
colors = [
    'red', 'green', 'blue', 'black', 'magenta', 'goldenrod', 'orange', 'purple',
    'brown', 'gray', 'navy', 'teal', 'coral', 'lime', 'indigo', 'darkgreen',
    'darkblue', 'darkred', 'salmon', 'chocolate'
]

In [ ]:
linestyles = [
    '-',        # solid
    '--',       # dashed
    '-.',       # dash-dot
    ':',        # dotted

    (0, (1, 1)),             # very dense dots
    (0, (2, 1)),             # dense dashed
    (0, (3, 1, 1, 1)),       # dash-dot-dot
    (0, (5, 1)),             # medium dashes
    (0, (5, 2)),             # spaced dashes
    (0, (4, 1, 2, 1)),       # dash-dot pattern
    (0, (3, 2, 1, 2)),       # dashed + dots
    (0, (2, 2, 2, 2)),       # equal segments
    (0, (6, 3)),             # long dashes
    (0, (1, 3)),             # short spaced dashes
    (0, (3, 3, 1, 1)),       # alternating patterns
    (0, (4, 4, 1, 1)),       # longer pattern
    (0, (7, 2, 1, 2)),       # long-short
    (0, (5, 5)),             # equally spaced long dashes
    (0, (1, 2, 3, 2)),       # complex pattern
    (0, (3, 1, 1, 1, 1, 1))  # more complex dash-dot
]

In [ ]:
def info(e):
    head   = list(e.keys())[0]
    body   = list(e[head].keys())
    bias   = e[head][body[0]]
    kernel = e[head][body[1]]
    return  head, body, list(bias), list(kernel)
def real(c):
    return float(np.real(c))  
def img(c):
    return float(np.imag(c))    
def r_i(c):
    return real(c),img(c)  

def save_params(step, params, energy):
    trained_params_list.append(params.copy())
    parameters_list.append(energy.state.parameters.copy())
    iii.append(1)
    return True

In [ ]:
def plot3(e_path1, e_path2, e_path3,
          j_out1, j_out2,j_out3,
          r_out1, r_out2,r_out3,
          f_out1, f_out2,f_out3,          
          L, IT, l_info, x_pos, y_pos,
          str_nets, show=False, stride = 5):
    
    fig, axs = plt.subplots(3, 1, figsize=(8, 6), sharex=True)

    j_out1 = j_out1 + ".log"
    r_out1 = r_out1 + ".log"
    f_out1 = f_out1 + ".log"
    
    j_out2 = j_out2 + ".log"
    r_out2 = r_out2 + ".log"
    f_out2 = f_out2 + ".log"
    
    j_out3 = j_out3+ ".log"
    r_out3 = r_out3 + ".log"
    f_out3 = f_out3 + ".log"
       

    # ========== Subplot (a) ==========
    ax = axs[0]


    df = pd.read_csv(e_path1)
    exact_gs_energy = df.iloc[0, 2]


    fx = j_out1
    with open(fx) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]

    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  
   
    fx = f_out1
    with open(fx) as f:
        data = json.load(f)
    
    iters_RBM   = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    mean_data = data["Energy"]["Mean"]
    if isinstance(mean_data, dict):
        energy_RBM = mean_data.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM = mean_data        
    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)


    fx = r_out1
    with open(fx) as f:
        data = json.load(f)
    
    iters_RBM2   = data["Energy"]["iters"]
    energy_raw2    = data["Energy"]["Mean"]
    mean_data2 = data["Energy"]["Mean"]
    if isinstance(mean_data2, dict):
        energy_RBM2 = mean_data2.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM2 = mean_data2        
    iters_RBM_ds2, energy_RBM_ds2 = downsample(iters_RBM2, energy_RBM2, stride)
    

    label = str_nets
    i = 0
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    i = 1
    label = r"JASTROW"
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)

    
    label = "RBM"
    i = 2
    ax.plot(iters_RBM_ds2, energy_RBM_ds2, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    if (exact_gs_energy != 0):
        ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label=f'Exact ({exact_gs_energy:.3f})')

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', 
                   length=2, width=0.8, top=True, bottom=True, 
                   left=True, right=True)
    ax.tick_params(labelbottom=False)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    #ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    ax.yaxis.set_minor_locator(FixedLocator([]))
    ax.set_ylabel(r"Energia")
    0.07
    ax.text(x_pos, y_pos, '(a) $L=' + str(L) + '$ - Regime '  + l_info[0], transform=ax.transAxes, 
            fontsize=10, verticalalignment='top')
    ax.legend(fontsize=10, loc='upper left',bbox_to_anchor=(1.02, 1), frameon=True)

    # ========== Subplot (b) ==========
    ax = axs[1]
    df = pd.read_csv(e_path2)
    exact_gs_energy = df.iloc[0, 2]

    fx = j_out2
    with open(fx) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  
    fx = f_out2
    with open(fx) as f:
        data = json.load(f)
    
    iters_RBM   = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    mean_data = data["Energy"]["Mean"]
    if isinstance(mean_data, dict):
        energy_RBM = mean_data.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM = mean_data
        
    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)

    fx = r_out2
    with open(fx) as f:
        data = json.load(f)
    
    iters_RBM2   = data["Energy"]["iters"]
    energy_raw2    = data["Energy"]["Mean"]
    mean_data2 = data["Energy"]["Mean"]
    if isinstance(mean_data2, dict):
        energy_RBM2 = mean_data2.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM2 = mean_data2       
        
    iters_RBM_ds2, energy_RBM_ds2 = downsample(iters_RBM2, energy_RBM2, stride)
    

    label = str_nets
    i = 0
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    i = 1
    label = r"JASTROW"
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)

    i = 2
    label = r"RBM"
    ax.plot(iters_RBM_ds2, energy_RBM_ds2, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)

    if (exact_gs_energy!=0):
            ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label=f'Exact ({exact_gs_energy:.3f})')

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    ax.set_ylabel(r"Energia")    
    #ax.set_xlabel(r"Interações")
    ax.text(x_pos, y_pos, '(b) $L=' + str(L) + '$  - Regime '  + l_info[1], 
            transform=ax.transAxes, fontsize=10, verticalalignment='top')
    ax.legend(fontsize=10, loc='upper left',bbox_to_anchor=(1.02, 1), frameon=True)


     # ========== Subplot (c) ==========
    ax = axs[2]
    df = pd.read_csv(e_path3)
    exact_gs_energy = df.iloc[0, 2]

    fx = j_out3
    with open(fx) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  

    
    fx = f_out3
    with open(fx) as f:
        data = json.load(f)
    
    iters_RBM   = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    mean_data = data["Energy"]["Mean"]
    if isinstance(mean_data, dict):
        energy_RBM = mean_data.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM = mean_data
       
    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)

    fx = r_out3
    with open(fx) as f:
        data = json.load(f)
    
    iters_RBM2   = data["Energy"]["iters"]
    energy_raw2    = data["Energy"]["Mean"]
    mean_data2 = data["Energy"]["Mean"]
    if isinstance(mean_data2, dict):
        energy_RBM2 = mean_data2.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM2 = mean_data2        
    iters_RBM_ds2, energy_RBM_ds2 = downsample(iters_RBM2, energy_RBM2, stride)
    

    label = str_nets
    i = 0
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    i = 1
    label = r"JASTROW"
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)

    i = 2
    label = r"RBM"
    ax.plot(iters_RBM_ds2, energy_RBM_ds2, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)

    if (exact_gs_energy!=0):
            ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label=f'Exact ({exact_gs_energy:.3f})')

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    ax.set_ylabel(r"Energia")    
    #ax.set_xlabel(r"Interações")
    ax.text(x_pos, y_pos, '(b) $L=' + str(L) + '$ - Regime '  + l_info[2],            
            transform=ax.transAxes, fontsize=10, verticalalignment='top')
    ax.legend(fontsize=10, loc='upper left',bbox_to_anchor=(1.02, 1), frameon=True)
    
    # Layout final
    plt.subplots_adjust(hspace=0.1, wspace=0.1)
    pathfg = "fig/mf/EGS_L_"  + str(L) + "_" + str(L) + "_IT_" + str(IT) + "_"  + upper(str_nets) + ".png"
    print(pathfg)
    plt.savefig(pathfg, dpi=300, bbox_inches='tight')
    if show :
        plt.show()
    plt.close()

In [ ]:
def plot1(e_path1, j_out1, r_out1, L, IT, l_info, l_tp, x_pos, y_pos, str_nets):
    fig, ax = plt.subplots(1, 1, figsize=(8, 6), sharex=True)

    j_out1 = j_out1 + ".log"
    r_out1 = r_out1 + ".log"

    # ========== Plot exact energy ==========
    df = pd.read_csv(e_path1)
    exact_gs_energy = df.iloc[0, 2]

    # ========== Plot Jastrow data ==========
    with open(j_out1) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw = data["Energy"]["Mean"]

    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    stride = 5
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  

    # ========== Plot RBM data ==========
    with open(r_out1) as f:
        data = json.load(f)
    
    iters_RBM = data["Energy"]["iters"]
    mean_data = data["Energy"]["Mean"]
    
    if isinstance(mean_data, dict):
        energy_RBM = mean_data.get("real", None)
    else:
        energy_RBM = mean_data

    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)

    # ========== Plotting ==========
    # Plot RBM
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=l_info[0], 
            linestyle=linestyles[0], marker=markers[0], 
            color=colors[0], markersize=5)
    
    # Plot Jastrow
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label="JASTROW", 
            linestyle=linestyles[1], marker=markers[1], 
            color=colors[1], markersize=5)

    if exact_gs_energy != 0:
        ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label=f'Exact ({exact_gs_energy:.3f})')


    
    # Axis configurations
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, 
                   top=True, bottom=True, left=True, right=True)
    
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_minor_locator(FixedLocator([]))
    
    ax.set_ylabel("Energia")
    ax.set_xlabel("Interações")
    
    ax.text(x_pos, y_pos, f'(a) $L={L}$ {l_tp[0]}', 
            transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)

    # Save and show
    pathfg = f"fig/wi/EGS_L_{L}_IT_{IT}_{str_nets}.png"
    print(pathfg)
    plt.savefig(pathfg, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
def plot2(e_path1, e_path2, 
          j_out1,j_out2,
          r_out1,r_out2,
          f_out1,f_out2,
          nf_out1,nf_out2,
          L, IT, l_info, l_tp,x_pos, y_pos,
          str_nets):

    TLABEL = "FFNN"
    
    fig, axs = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

    j_out1 = j_out1 + ".log"
    j_out2 = j_out2 + ".log"
    
    r_out1 = r_out1 + ".log"
    r_out2 = r_out2 + ".log"

    f_out1 = f_out1 + ".log"
    f_out2 = f_out2 + ".log"

    nf_out1 = nf_out1 + ".log"
    nf_out2 = nf_out2 + ".log"

    # ========== Subplot (a) ==========
    ax = axs[0]


    df = pd.read_csv(e_path1)
    exact_gs_energy = df.iloc[0, 2]


    fx = j_out1
    with open(fx) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]

    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    stride = 5
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  
   
    fx = r_out1
    with open(fx) as f:
        data = json.load(f)
    
    iters_RBM   = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    mean_data = data["Energy"]["Mean"]
    if isinstance(mean_data, dict):
        energy_RBM = mean_data.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM = mean_data      
    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)


    f2x = f_out1
    with open(f2x) as f:
        data = json.load(f)
    
    iters_RBM2       = data["Energy"]["iters"]
    energy_raw2      = data["Energy"]["Mean"]
    mean_data2       = data["Energy"]["Mean"]
    if isinstance(mean_data2, dict):
        energy_RBM2 = mean_data2.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM2 = mean_data2      
    iters_RBM_ds2, energy_RBM_ds2 = downsample(iters_RBM2, energy_RBM2, stride)


    
    f3x = nf_out1
    with open(f3x) as f:
        data = json.load(f)
    
    iters_RBM3       = data["Energy"]["iters"]
    energy_raw3      = data["Energy"]["Mean"]
    mean_data3       = data["Energy"]["Mean"]
    if isinstance(mean_data3, dict):
        energy_RBM3 = mean_data3.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM3 = mean_data3      
    iters_RBM_ds3, energy_RBM_ds3 = downsample(iters_RBM3, energy_RBM3, stride)


    ltp = l_info[0]
    
    i = 0
    label = r"JASTROW"
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)

   
    label = l_info[0]
    i = 1
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
  
    label = l_info[1]
    i = 2
    ax.plot(iters_RBM_ds2, energy_RBM_ds2, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = l_info[2]
    i = 3
    ax.plot(iters_RBM_ds3, energy_RBM_ds3, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)



    if (exact_gs_energy != 0):
        ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label=f'Exact ({exact_gs_energy:.3f})')

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', 
                   length=2, width=0.8, top=True, bottom=True, 
                   left=True, right=True)
    ax.tick_params(labelbottom=False)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    #ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    ax.yaxis.set_minor_locator(FixedLocator([]))
    ax.set_ylabel(r"Energia")
    0.07
    ax.text(x_pos, y_pos, '(a) $ L=' + str(L)  + '$, '+ l_tp[0], transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)





    
    # ========== Subplot (b) ==========
    ax = axs[1]
    df = pd.read_csv(e_path2)
    exact_gs_energy = df.iloc[0, 2]

    fx = j_out2
    with open(fx) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    stride = 5
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  
    fx = r_out2
    with open(fx) as f:
        data = json.load(f)
    
    iters_RBM   = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    mean_data = data["Energy"]["Mean"]
    if isinstance(mean_data, dict):
        energy_RBM = mean_data.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM = mean_data
       
    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)


    f2x = f_out2
    with open(f2x) as f:
        data = json.load(f)
    
    iters_RBM2       = data["Energy"]["iters"]
    energy_raw2      = data["Energy"]["Mean"]
    mean_data2       = data["Energy"]["Mean"]
    if isinstance(mean_data2, dict):
        energy_RBM2 = mean_data2.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM2 = mean_data2      
    iters_RBM_ds2, energy_RBM_ds2 = downsample(iters_RBM2, energy_RBM2, stride)


    f3x = nf_out2
    with open(f3x) as f:
        data = json.load(f)
    
    iters_RBM3       = data["Energy"]["iters"]
    energy_raw3      = data["Energy"]["Mean"]
    mean_data3       = data["Energy"]["Mean"]
    if isinstance(mean_data3, dict):
        energy_RBM3 = mean_data3.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM3 = mean_data3      
    iters_RBM_ds3, energy_RBM_ds3 = downsample(iters_RBM3, energy_RBM3, stride)

    
    ltp = l_info[1]

    i = 0
    label = r"JASTROW"
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)

    label = l_info[0]
    i = 1
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
   

    label = l_info[1]
    i = 2
    ax.plot(iters_RBM_ds2, energy_RBM_ds2, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = l_info[2]
    i = 3
    ax.plot(iters_RBM_ds3, energy_RBM_ds3, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)



    if (exact_gs_energy!=0):
            ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label=f'Exact ({exact_gs_energy:.3f})')

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    ax.set_ylabel(r"Energia")    
    #ax.set_xlabel(r"Interações")
    ax.text(x_pos, y_pos, '(b) $ L=' + str(L) + '$, ' +  l_tp[1] , transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)
    
    
    # Layout final
    plt.subplots_adjust(hspace=0.1, wspace=0.1)
    pathfg = "fig/wff/EGS_L_" + str(L) + "_" + str(L) + "_IT_" + str(IT) + "_"  + str_nets + ".png"
    print(pathfg)
    plt.savefig(pathfg, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
def plot4(e_path1, e_path2, 
          j_out1, r_out1, 
          j_out2, r_out2, 
          e_path3, e_path4, 
          j_out3, r_out3, 
          j_out4, r_out4, 
          L1,L3,IT, l_info, l_tp,x_pos, y_pos,
          str_nets):

    TLABEL = "FFNN"
    
    fig, axs = plt.subplots(4, 1, figsize=(8, 6), sharex=True)

    j_out1 = j_out1 + ".log"
    r_out1 = r_out1 + ".log"
    j_out2 = j_out2 + ".log"
    r_out2 = r_out2 + ".log"
    j_out3 = j_out3 + ".log"
    r_out3 = r_out3 + ".log"
    j_out4 = j_out4 + ".log"
    r_out4 = r_out4 + ".log"
    

    # ========== Subplot (a) ==========
    ax = axs[0]


    df = pd.read_csv(e_path1)
    exact_gs_energy = df.iloc[0, 2]


    fx = j_out1
    with open(fx) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]

    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    stride = 5
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  
   
    fx = r_out1
    with open(fx) as f:
        data = json.load(f)
    
    iters_RBM   = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    mean_data = data["Energy"]["Mean"]
    if isinstance(mean_data, dict):
        energy_RBM = mean_data.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM = mean_data

    
    
    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)
    ltp = l_info[0]


    label = l_info[0]
    i = 0
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    i = 1
    label = r"JASTROW"
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)


    if (exact_gs_energy != 0):
        ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label=f'Exact ({exact_gs_energy:.3f})')

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', 
                   length=2, width=0.8, top=True, bottom=True, 
                   left=True, right=True)
    ax.tick_params(labelbottom=False)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    #ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    ax.yaxis.set_minor_locator(FixedLocator([]))
    ax.set_ylabel(r"Energia")
    0.07
    ax.text(x_pos, y_pos, '(a) $L=' + str(L1) + '$ ' + l_tp[0] , transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)

    # ========== Subplot (b) ==========
    ax = axs[1]
    df = pd.read_csv(e_path2)
    exact_gs_energy = df.iloc[0, 2]

    fx = j_out2
    with open(fx) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    stride = 5
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  
    fx = r_out2
    with open(fx) as f:
        data = json.load(f)
    
    iters_RBM   = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    mean_data = data["Energy"]["Mean"]
    if isinstance(mean_data, dict):
        energy_RBM = mean_data.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM = mean_data


        
    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)
    ltp = l_info[1]




    label = l_info[1]
    i = 0
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    i = 1
    label = r"JASTROW"
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)


    if (exact_gs_energy!=0):
            ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label=f'Exact ({exact_gs_energy:.3f})')

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    ax.set_ylabel(r"Energia")    
    #ax.set_xlabel(r"Interações")
    ax.text(x_pos, y_pos, '(b) $L=' + str(L1) + '$ ' + l_tp[1] , transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)




    # ========== Subplot (c) ==========
    ax = axs[2]                   
    df = pd.read_csv(e_path3)
    exact_gs_energy = df.iloc[0, 2]

    fx = j_out3
    with open(fx) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    stride = 5
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  
    fx = r_out3
    with open(fx) as f:
        data = json.load(f)
    

    iters_RBM   = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    mean_data = data["Energy"]["Mean"]
    if isinstance(mean_data, dict):
        energy_RBM = mean_data.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM = mean_data
    
    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)
    ltp =l_info[2]

    stride = 5


    label = l_info[2]
    i = 0
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    i = 1
    label = r"JASTROW"
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)


    if (exact_gs_energy!=0):
            ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label=f'Exact ({exact_gs_energy:.3f})')

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.set_ylabel(r"Energia")    
    #ax.set_xlabel(r"Interações")
    ax.text(x_pos, y_pos, '(c) $L=' + str(L3) + '$ ' + l_tp[2], transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)



    # ========== Subplot (d) ==========
    ax = axs[3]
    df = pd.read_csv(e_path4)
    exact_gs_energy = df.iloc[0, 2]

    fx = j_out4
    with open(fx) as f:
        data = json.load(f)

    iters_Jastrow = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    if isinstance(energy_raw, list):
        energy_Jastrow = [e["real"] for e in energy_raw]
    else:
        energy_Jastrow = energy_raw["real"]

    stride = 5
    iters_Jastrow_ds, energy_Jastrow_ds = downsample(iters_Jastrow, energy_Jastrow, stride)  
    fx = r_out4
    with open(fx) as f:
        data = json.load(f)
    
    
    
    iters_RBM   = data["Energy"]["iters"]
    energy_raw    = data["Energy"]["Mean"]
    mean_data = data["Energy"]["Mean"]
    if isinstance(mean_data, dict):
        energy_RBM = mean_data.get("real", None)  # ou algum valor padrão
    else:
        energy_RBM = mean_data  
   
    iters_RBM_ds, energy_RBM_ds = downsample(iters_RBM, energy_RBM, stride)
    ltp = l_info[3]

    stride = 5

    label = l_info[3]
    i = 0
    ax.plot(iters_RBM_ds, energy_RBM_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    i = 1
    label = r"JASTROW"
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)


    if (exact_gs_energy!=0):
            ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label=f'Exact ({exact_gs_energy:.3f})')

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    ax.set_ylabel(r"Energia")    
    ax.set_xlabel(r"Interações")
    ax.text(x_pos, y_pos, '(d) $L=' + str(L3) + '$ ' + l_tp[3], transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)

    
    # Layout final
    plt.subplots_adjust(hspace=0.1, wspace=0.1)
    pathfg = "fig/w/EGS_L_" + str(L1) + "_" + str(L3) + "_IT_" + str(IT) + "_"  + str_nets + ".png"
    print(pathfg)
    plt.savefig(pathfg, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
def plotmw(paths1,paths2,L):
    
    fig, axs = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

    # Aumentar o espaço à esquerda para acomodar a legenda externa
    fig.subplots_adjust(left=0.12, right=0.75, hspace=0.25)

    df1   = pd.read_csv(paths1[2])
    ncol1 = len(df1.columns) - 2

    df2   = pd.read_csv(paths2[2])
    ncol2 = len(df2.columns) - 2


    x1 = df1["id"]
    x2 = df2["id"]

    # ========== Subplot (a) ==========
    ax = axs[0]
    stride = 5
    i = 0
    label = r"avg (w)"
    m1 = df1.iloc[:, 1:].mean(axis=1).round(2) 
    i  = i + 1 
    x1_ds, m1_ds = downsample(x1, m1, stride)
    ax.plot(x1_ds, m1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"median (w)"
    median1 = df1.iloc[:, 1:].median(axis=1).round(2)
    i  = i + 1 
    x1_ds, median1_ds = downsample(x1, median1, stride)
    ax.plot(x1_ds, median1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"std (w)"
    std1 = df1.iloc[:, 1:].std(axis=1).round(2)
    i  = i + 1 
    x1_ds, std1_ds = downsample(x1, std1, stride)
    ax.plot(x1_ds, std1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"log(amp) (w)"
    range1 = (df1.iloc[:, 1:].max(axis=1) - df1.iloc[:, 1:].min(axis=1)).round(2)
    range1 = np.log10(range1)  
    i  = i + 1 
    x1_ds, range1_ds = downsample(x1, range1, stride)
    ax.plot(x1_ds, range1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"log(disp) (w)"
    cv1 = (df1.iloc[:, 1:].std(axis=1) / df1.iloc[:, 1:].mean(axis=1)).round(2)
    cv1 = np.log10(cv1)        # Log base 10

    i  = i + 1 
    x1_ds, cv1_ds = downsample(x1, cv1, stride)
    ax.plot(x1_ds, cv1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    all_y_values = np.concatenate([
        m1_ds,
        m1_ds + std1_ds,  # Limite superior do desvio padrão
        m1_ds - std1_ds,  # Limite inferior do desvio padrão
        median1_ds,
        range1_ds
    ])



    y_padding = 0.1 * (np.nanmax(all_y_values) - np.nanmin(all_y_values))
    y_min = np.nanmin(all_y_values) - y_padding
    y_max = np.nanmax(all_y_values) + y_padding

    ax.set_ylim(y_min, y_max)


    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, 
                   bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, 
                   width=0.8, top=True, bottom=True, left=True, right=True)
    ax.tick_params(labelbottom=False)

    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_minor_locator(FixedLocator([]))
   
    
    ax.set_ylabel(r"Metrics Weights")

    
    # Mover o texto (a) para cima do quadro do gráfico
    ax.text(0.03, 1.0, '(a) $L=' + str(L) + '$ Regime Ferromagnético', 
            transform=ax.transAxes, fontsize=12, verticalalignment='bottom') 
    
    # Mover a legenda para fora à esquerda
    ax.legend(fontsize=9, loc='upper right', bbox_to_anchor=(1.28, 1), 
              frameon=True, edgecolor = 'black')
    

    # ========== Subplot (c) ==========
    x1 = x2
    ax = axs[1]
    i = 0
    label = r"avg (w)"
    m1 = df2.iloc[:, 1:].mean(axis=1).round(2) 
    i  = i + 1 
    x1_ds, m1_ds = downsample(x1, m1, stride)
    ax.plot(x1_ds, m1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"median (w)"
    median1 = df2.iloc[:, 1:].median(axis=1).round(2)
    i  = i + 1 
    x1_ds, median1_ds = downsample(x1, median1, stride)
    ax.plot(x1_ds, median1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"std (w)"
    std1 = df2.iloc[:, 1:].std(axis=1).round(2)
    i  = i + 1 
    x1_ds, std1_ds = downsample(x1, std1, stride)
    ax.plot(x1_ds, std1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"log(amp) (w)"
    range1 = (df2.iloc[:, 1:].max(axis=1) - df2.iloc[:, 1:].min(axis=1)).round(2)
    range1 = np.log10(range1)   
    
    i  = i + 1 
    x1_ds, range1_ds = downsample(x1, range1, stride)
    ax.plot(x1_ds, range1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    label = r"log(disp) (w)"
    cv1 = (df2.iloc[:, 1:].std(axis=1) / df2.iloc[:, 1:].mean(axis=1)).round(2)
    cv1 = np.log10(cv1)        # Log base 10

    i  = i + 1 
    x1_ds, cv1_ds = downsample(x1, cv1, stride)
    ax.plot(x1_ds, cv1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)

    # Configurações do eixo

    all_y_values = np.concatenate([
        m1_ds,
        m1_ds + std1_ds,  # Limite superior do desvio padrão
        m1_ds - std1_ds,  # Limite inferior do desvio padrão
        median1_ds,
        range1_ds
    ])

    y_padding = 0.1 * (np.nanmax(all_y_values) - np.nanmin(all_y_values))
    y_min = np.nanmin(all_y_values) - y_padding
    y_max = np.nanmax(all_y_values) + y_padding

    ax.set_ylim(y_min, y_max)
    
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.set_ylabel(r"Metrics Weights")
    ax.set_xlabel(r"Interações")

    # Mover o texto (b) para cima do quadro do gráfico
    ax.text(0.03, 1.0, '(b) $L=' + str(L) + '$ Regime Antiferromagnético', 
            transform=ax.transAxes, fontsize=12, verticalalignment='bottom')
    
    # Mover a legenda para fora à esquerda
    ax.legend(fontsize=9, loc='upper right', bbox_to_anchor=(1.28, 1), 
              frameon=True, framealpha = 1, edgecolor = 'black')
    

    # Layout final
    plt.subplots_adjust(hspace=0.1, wspace=0.1)
    plt.savefig("fig/wi/E_W_L_M_" + str(L) + "0_1.png", dpi=300, bbox_inches='tight')
    #plt.show()
    plt.close()

In [ ]:
def plotw(paths1,paths2,L, kb):
    
    fig, axs = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

    # Aumentar o espaço à esquerda para acomodar a legenda externa
    fig.subplots_adjust(left=0.12, right=0.75, hspace=0.25)

    df1   = pd.read_csv(paths1[kb])
    df2   = pd.read_csv(paths2[kb])

    print(paths1[kb],paths2[kb])

    #kb = 2; kernel
    if kb == 0 or kb == 1:
        ncol1 = 1
        ncol2 = 1
    else:
        ncol1 = len(df1.columns) - 2
        ncol2 = len(df2.columns) - 2
        

    # ========== Subplot (a) ==========
    ax = axs[0]
    stride = 5

    x1 = df1["id"]
    
    if kb == 0 or kb == 1:
        index = 0
        i = 0
        w1 = df1['0']
        x1_ds, w1_ds = downsample(x1, w1, stride) 
        
        label = r"bias" 
        ax.plot(x1_ds, w1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    else:
        index = 0
        for i in range(0,10):
            nn = random.randint(1, ncol1) 
            w1 = df1[str(nn)]
            x1_ds, w1_ds = downsample(x1, w1, stride) 
            index = i + 1
            label = r"w" + str(index)
            ax.plot(x1_ds, w1_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)
   
    
    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, 
                   bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, 
                   width=0.8, top=True, bottom=True, left=True, right=True)
    ax.tick_params(labelbottom=False)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_minor_locator(FixedLocator([]))

    if kb == 0 or kb == 1:
       ax.set_ylabel(r"Bias")
    else:
        ax.set_ylabel(r"Weights")
    
    

    
    # Mover o texto (a) para cima do quadro do gráfico
    ax.text(0.03, 1.0, '(a) $L=' + str(L) + '$ Regime Ferromagnético', 
            transform=ax.transAxes, fontsize=12, verticalalignment='bottom') 
    
    # Mover a legenda para fora à esquerda
    ax.legend(fontsize=9, loc='upper right', bbox_to_anchor=(1.28, 1), 
              frameon=True, edgecolor = 'black')
    

    # ========== Subplot (c) ==========
    ax = axs[1]


    x2 = df2["id"]

    if kb == 0 or kb == 1:
        index = 0
        i = 0
        w1 = df2['0']
        x1_ds, w1_ds = downsample(x1, w1, stride) 
        
        label = r"bias" 
        ax.plot(x1_ds, w1_ds, label=label, linestyle=linestyles[i],
            marker=markers[i], color=colors[i], markersize=5)
    else:
        
        index = 0
        for i in range(0,10):
            nn = random.randint(1, ncol2) 
            w2 = df2[str(nn)]
            x2_ds, w2_ds = downsample(x2, w2, stride) 
            index = i + 1
            label = r"w" + str(index)
            ax.plot(x2_ds, w2_ds, label=label, linestyle=linestyles[i],
                marker=markers[i], color=colors[i], markersize=5)
           
    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_major_locator(MaxNLocator(integer=True))

    if kb == 0 or kb == 1:
       ax.set_ylabel(r"Bias")
    else:
        ax.set_ylabel(r"Weights")

    ax.set_xlabel(r"Interações")

    # Mover o texto (b) para cima do quadro do gráfico
    ax.text(0.03, 1.0, '(b) $L=' + str(L) + '$ Regime Antiferromagnético', 
            transform=ax.transAxes, fontsize=12, verticalalignment='bottom')
    
    # Mover a legenda para fora à esquerda
    ax.legend(fontsize=9, loc='upper right', bbox_to_anchor=(1.28, 1), 
              frameon=True, framealpha = 1, edgecolor = 'black')
    

    # Layout final
    plt.subplots_adjust(hspace=0.1, wspace=0.1)
    plt.savefig("fig/wi/E_W_L_" + str(L) +  
                "_KB_" + str(kb) +  "_0_1.png", dpi=300, bbox_inches='tight')
    #plt.show()
    plt.close()

In [ ]:
def get_hist_max(values):
    values = values.values.astype(float)
    values = np.nan_to_num(values, nan=0.0)  # Substituir NaNs por 0
    hist, _ = np.histogram(values, bins=15, density=True)
    return hist.max()

def plot_with_normal_w(ax, data, color, title=''):
    import numpy as np
    import scipy.stats as stats

    values = data.values.astype(float)
    mean, std = values.mean(), values.std()

    # Histograma normalizado
    ax.hist(values, bins=15, color=color, alpha=0.6, density=True)

    # Curva da distribuição normal
    xmin, xmax = ax.get_xlim()
    x = np.linspace(xmin, xmax, 100)
    p = stats.norm.pdf(x, mean, std)
    ax.plot(x, p, 'k', linewidth=1.5)

    # Título opcional
    if title:
        ax.set_title(title, fontsize=10)

    # Ajusta o limite y automaticamente baseado no pico da PDF
    p_max = stats.norm.pdf(mean, loc=mean, scale=std)
    ax.set_ylim(0, p_max * 1.1)  # 10% de margem
    
def plot_with_normal(ax, data, color, title, y_max):
    import scipy.stats as stats
    import numpy as np
    
    try :
        values = data.values.astype(float)
        values[~np.isfinite(values)] = 0.0
    except :
        print("erro")

     
    mean, std = values.mean(), values.std()
    
    ax.hist(values, bins=15, color=color, alpha=0.6, density=True)
    xmin, xmax = ax.get_xlim()
    x = np.linspace(xmin, xmax, 100)
    p = stats.norm.pdf(x, mean, std)
    ax.plot(x, p, 'k', linewidth=1.5)
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, y_max)

def plot_dist(path1, path2, ip, lp,L,it,va,y_max):
    
    df1 = pd.read_csv(path1[2])
    df2 = pd.read_csv(path2[2])

    df1.fillna(0, inplace=True)
    df2.fillna(0, inplace=True)

    lnw1 = df1.shape[1]
    lnw2 = df1.shape[1]


    i_line1 = df1.iloc[ip].iloc[1:]  
    l_line1 = df1.iloc[lp].iloc[1:]  

    i_line2 = df2.iloc[ip].iloc[1:]  
    l_line2 = df2.iloc[lp].iloc[1:]


    fig, axs = plt.subplots(1, 3, figsize=(10, 4), sharex=True)

    # Plot 1 - Distribuição Inicial
    plot_with_normal(axs[0], i_line1, 'orange', '(a) Inicial',y_max )

    axs[0].set_ylim(0, y_max)
    axs[0].set_xticks([])
    axs[0].set_xlabel('')

    axins = axs[0].inset_axes([0.1, 0.5, 0.45, 0.4])
    plot_with_normal_w(axins, i_line1, 'orange')
    axins.set_xticks([])
    axins.set_yticks([])

    # Plot 2 - Distribuição Final
    plot_with_normal(axs[1], l_line1, 'crimson', '(b) Ferromagnetismo',y_max)
    axs[1].set_ylim(0, y_max)
    axs[1].set_xticks([])
    axs[1].set_xlabel('')

    # Plot 3 - Distribuição Final - Dataset 2
    plot_with_normal(axs[2], l_line2, 'darkblue', '(b) Antiferromagnetismo',y_max)
    axs[2].set_ylim(0, y_max)
    axs[2].set_xticks([])
    axs[2].set_xlabel('')

    plt.tight_layout()

    pathimg = "distf/dif_dist_" +  str(L) +  '_' + str(it) + '_' + str(va)  
    plt.savefig(pathimg, dpi=300, bbox_inches='tight')

    
    plt.show()
    plt.close()


In [ ]:
def plot_dif(path1, path2, ip, lp):
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt

    df1 = pd.read_csv(path1[2])
    df2 = pd.read_csv(path2[2])

    df1.fillna(0, inplace=True)
    df2.fillna(0, inplace=True)


    i_line1 = df1.iloc[ip].iloc[1:]  
    l_line1 = df1.iloc[lp].iloc[1:] 
    
    i_line2 = df2.iloc[ip].iloc[1:]  
    l_line2 = df2.iloc[lp].iloc[1:]

    dif1 = l_line1 - i_line1
    dif2 = l_line2 - i_line2

    fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharex=True)

    for idx, (ax, dif, title) in enumerate(zip(axs, [dif1, dif2], ['Dataset 1', 'Dataset 2'])):
        bars = ax.bar(range(len(dif)), dif.values,
                      color=np.where(dif.values >= 0, 'green', 'red'),
                      alpha=0.6, edgecolor='black')

        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_title(f'Diferenças {title}: Final - Inicial', fontsize=12)
        ax.set_xlabel('Índice da Variável')
        ax.set_ylabel('Diferença')
        ax.grid(axis='y', alpha=0.3)

        # Define rótulos numéricos apenas em alguns ticks
        ax.set_xticks([])

    plt.tight_layout()
    plt.show()
    plt.close()


In [ ]:
def plotdp(paths1, paths2, initial_point_idx, last_point_idx, 
           L, IT, str_nets):

    pathfg = "fig/wi/PCA_" + str(L) + "_" + str(L) + "_IT_" + str(IT) + "_"  + str_nets + ".png"


    try:
        # Data loading and validation
        df1 = pd.read_csv(paths1[2])
        if df1.empty:
            raise ValueError("Dataframe 1 is empty")

        df2 = pd.read_csv(paths2[2])
        if df2.empty:
            raise ValueError("Dataframe 2 is empty")

        # Create figure with two subplots (vertical arrangement)
        fig, axs = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
        
        # Process and plot first dataset (Ferromagnético)
        plot_pca(axs[0], df1, initial_point_idx, last_point_idx, 
                title='(a) Regime Ferromagnético',
                point_labels=['Ponto Inicial', 'Ponto Final'])
        
        # Process and plot second dataset (Antiferromagnético)
        plot_pca(axs[1], df2, initial_point_idx, last_point_idx,
                title='(b) Regime Antiferromagnético',
                point_labels=['Ponto Inicial', 'Ponto Final'])

        # Configurações finais do layout
        plt.subplots_adjust(hspace=0.3, wspace=0.1)  # Aumentei o hspace para 0.3
        plt.savefig(pathfg, dpi=300, bbox_inches='tight')
        plt.show()
        
        plt.close()
        print(f"Figure saved to {pathfg}")
        
    except Exception as e:
        print(f"Error in plot_dynamics: {str(e)}")
        raise

def plot_pca(ax, df, initial_point_idx, last_point_idx, title, point_labels):
    # Extract points
    all_points = df.iloc[:, 1:].values  # Todas as séries (sem a primeira coluna)

    # Trata valores ausentes (NaN) com média da coluna
    imputer = SimpleImputer(strategy="mean")
    all_points_imputed = imputer.fit_transform(all_points)

    # Padronização (zero média, desvio padrão 1)
    X_std = StandardScaler().fit_transform(all_points_imputed)

    # PCA
    n_components = min(2, X_std.shape[1])
    pca = PCA(n_components=n_components)
    principal_components = pca.fit_transform(X_std)

    # Distância Euclidiana entre o ponto inicial e final no espaço PCA
    distance = np.linalg.norm(
        principal_components[initial_point_idx] - principal_components[last_point_idx]
    )

    # Configurações do estilo do gráfico
    colors = ['green', 'red']
    mark = ['o', 's']
    line_style = 'grey'
    alpha = 0.3

    # Plot PCA Trajectory
    if n_components >= 2:
        # Full trajectory
        ax.scatter(principal_components[:, 0], principal_components[:, 1], 
                  alpha=alpha, color='blue', marker='.', label='Pontos Intermediários')
        ax.plot(principal_components[:, 0], principal_components[:, 1], 
               color=line_style, alpha=alpha, linestyle='-', label='Trajetória')
        
        # Highlight points
        ax.scatter(principal_components[initial_point_idx, 0], 
                  principal_components[initial_point_idx, 1], 
                  color=colors[0], s=100, marker=mark[0], label=point_labels[0])
        ax.scatter(principal_components[last_point_idx, 0], 
                  principal_components[last_point_idx, 1], 
                  color=colors[1], s=100, marker=mark[1], label=point_labels[1])
        
        ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    else:
        # 1D case
        ax.scatter(principal_components[:, 0], np.zeros(len(principal_components)), 
                  alpha=alpha, color='blue', marker='.', label='Pontos Intermediários')
        ax.plot(principal_components[:, 0], np.zeros(len(principal_components)), 
               color=line_style, alpha=alpha, linestyle='-', label='Trajetória')
        ax.scatter(principal_components[initial_point_idx, 0], 0, 
                  color=colors[0], s=100, marker=mark[0], label=point_labels[0])
        ax.scatter(principal_components[last_point_idx, 0], 0, 
                  color=colors[1], s=100, marker=mark[1], label=point_labels[1])

    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    
    # Configurações dos labels
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    
    # Título e legenda
    ax.set_title(title, fontsize=12, loc='left', pad=10)
    ax.legend(fontsize=9, loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)  # Legenda fora
    ax.grid(alpha=0.3)
    
    # Ajusta o layout para acomodar a legenda
    plt.tight_layout(rect=[0, 0, 0.85, 1])  # Reduz a área útil em 15% à direita para a legenda

In [ ]:
def plot_distributions(paths1, paths2, initial_point_idx, last_point_idx, 
                       L, IT, str_nets):

    pathfg = "fig/wi/NDP_" + str(L) + "_" + str(L) + "_IT_" + str(IT) + "_"  + str_nets + ".png"


    try:
        # Carregar os dados
        df1 = pd.read_csv(paths1[2])
        df2 = pd.read_csv(paths2[2])

        # Criar figura com 1 linha e 2 colunas
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

        # Plotar distribuição Ferromagnética
        plot_kde_comparison(ax1, df1, initial_point_idx, last_point_idx,
                          'Distribuição Ferromagnética')

        # Plotar distribuição Antiferromagnética
        plot_kde_comparison(ax2, df2, initial_point_idx, last_point_idx,
                          'Distribuição Antiferromagnética')

        # Ajustes finais
        plt.tight_layout()
        plt.savefig(pathfg, dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f"Figura salva como {pathfg}")

    except Exception as e:
        print(f"Erro: {str(e)}")
        raise

def plot_kde_comparison(ax, df, initial_idx, last_idx, title):
    # Extrair os pontos
    initial = df.iloc[initial_idx].iloc[1:].values
    final = df.iloc[last_idx].iloc[1:].values

    # Plotar KDE
    sns.kdeplot(initial, color='orange', label='Inicial', fill=True, alpha=0.3, ax=ax)
    sns.kdeplot(final, color='crimson', label='Final', fill=True, alpha=0.3, ax=ax)

    # Linhas de média
    ax.axvline(initial.mean(), color='orange', linestyle='--', 
              label=f'Média Inicial: {initial.mean():.2f}')
    ax.axvline(final.mean(), color='crimson', linestyle='--',
              label=f'Média Final: {final.mean():.2f}')

    # Configurações do gráfico
    ax.set_title(title, fontsize=12, pad=10)
    ax.set_xlabel('Valores', fontsize=10)
    ax.set_ylabel('Densidade', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)
    ax.tick_params(direction='in', which='both')

In [ ]:
def plot_pearson_neurons(paths1, paths2, point_idx, L, IT, str_nets, lit):
    
    i = 0
    
    for p1,p2 in  zip(paths1, paths2):

        pathfg = "fig/mf/PRSN_" + str(L) + "_" + str(L) + "_IT_" + str(IT) + "_"  + str_nets + "_" + str(lit) 

        print(p1,p2)

      
        df1 = pd.read_csv(p1)
        df2 = pd.read_csv(p2)


        
        if len(df1) > 0 :
            
            df1 = df1.iloc[:, 1:]  
            df2 = df2.iloc[:, 1:]
        
            # Calcular correlações para cada regime
            corr1 = df1.corr(method="pearson")
            corr2 = df2.corr(method="pearson")    
       
            corr_metrics(df1, 0, L,IT,str_nets, lit, i)

            corr_metrics(df2, 1, L,IT,str_nets, lit, i)

            # --- Plot ---
            fig, axs = plt.subplots(1, 2, figsize=(14, 6))

        
            sns.heatmap(corr1,   
                        cmap="coolwarm",  vmin=-1, vmax=1, square=True, annot=False, ax=axs[0])
            axs[0].set_title( '(a) $L=' + str(L) + '$ - Regime Ferromagnético')
            axs[0].set_xlabel('Neurônios', fontsize=10)
            axs[0].set_ylabel('Neurônios', fontsize=10)   
            axs[0].legend(fontsize=10)

            sns.heatmap(corr2, 
                        cmap="coolwarm",
                        vmin=-1, vmax=1, square=True, annot=False, ax=axs[1])
            axs[1].set_title( '(b) $L=' + str(L) + '$ - Regime Antiferromagnético')
            axs[1].set_xlabel('Neurônios', fontsize=10)
            axs[1].set_ylabel('')   

            plt.tight_layout()

            i = i + 1
        
            pathfg = pathfg +  str(i) + ".png"
    
            plt.savefig(pathfg, dpi=300)
            plt.close(fig) 

            print(f"Figura salva em: {pathfg}")

In [ ]:
def _plot_pearson_neurons(paths1, paths2, point_idx, L, IT, str_nets):
    
    pathfg = "fig/wff/PRSN_" + str(L) + "_" + str(L) + "_IT_" + str(IT) + "_"  + str_nets + ".png"

  
    def prepare_corr_matrix(path, idx):
        df = pd.read_csv(path)
        data = df.iloc[idx, 1:].astype(float)  # remove 'id'
        df_neurons = pd.DataFrame([data.values], columns=[f"N{i+1}" for i in range(len(data))])
        return df_neurons.T.corr()  # correlação entre todos os neurônios

    # Calcular correlações para cada regime
    corr1 = pd.read_csv(paths1[2]).iloc[:, 1:].corr(method="pearson")
    corr2 = pd.read_csv(paths2[2]).iloc[:, 1:].corr(method="pearson")

    # --- Plot ---
    fig, axs = plt.subplots(1, 2, figsize=(14, 6))

    sns.heatmap(corr1, cmap=sns.diverging_palette(220, 10, as_cmap=True),
                vmin=-1, vmax=1, square=True, annot=False, ax=axs[0])
    axs[0].set_title("Ferromagnetismo")

    sns.heatmap(corr2, cmap=sns.diverging_palette(220, 10, as_cmap=True),
                vmin=-1, vmax=1, square=True, annot=False, ax=axs[1])
    axs[1].set_title("Antiferromagnetismo")

    plt.tight_layout()
    plt.savefig(pathfg, dpi=300)
    plt.show()

    print(f"Figura salva em: {pathfg}")

In [ ]:
def plot_corr_sum(df):
    """
    Plota corr_sum em função de w com 4 faixas desconectadas:
      - 0 a 22     -> verde
      - 23 a 160   -> laranja  
      - 161 a 270  -> rosado
      - 271 a 359  -> verde (mas desconectado da primeira faixa verde)
    """
    plt.figure(figsize=(8, 5))

    # Faixa 1: 0–22 (verde - antiferromagnético)
    mask1 = (df["w"] >= 0) & (df["w"] <= 22)
    plt.plot(df["w"][mask1], df["corr_sum"][mask1],
             marker="o", linestyle="-", color="green")

    # Faixa 2: 23–160 (laranja - frustrado)
    mask2 = (df["w"] >= 23) & (df["w"] <= 160)
    plt.plot(df["w"][mask2], df["corr_sum"][mask2],
             marker="s", linestyle="-", color="orange")

    # Faixa 3: 161–270 (rosado - ferromagnético)
    mask3 = (df["w"] >= 161) & (df["w"] <= 270)
    plt.plot(df["w"][mask3], df["corr_sum"][mask3],
             marker="^", linestyle="-", color="crimson")

    # Faixa 4: 271–359 (verde - antiferromagnético, mas desconectado)
    mask4 = (df["w"] >= 271) & (df["w"] <= 359)
    if mask4.any():
        plt.plot(df["w"][mask4], df["corr_sum"][mask4],
                 marker="o", linestyle="-", color="green")

    # Adicionar valores sobre os pontos
    for x, y in zip(df["w"], df["corr_sum"]):
        plt.text(x, y + 0.02, f"{y:.2f}", ha="center", fontsize=7, color="black")

    # Criar legenda personalizada sem repetições
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='green', linestyle='-', 
               label='antiferromagnético'),
        Line2D([0], [0], marker='s', color='orange', linestyle='-', 
               label='frustrado'),
        Line2D([0], [0], marker='^', color='crimson', linestyle='-', 
               label='ferromagnético')
    ]
    
    plt.legend(handles=legend_elements, frameon=False, fontsize=10)

    # Estilo científico
    plt.xticks(range(0, 359, 30))  # 0,30,60,...,330
    plt.xlabel(r"$w$", fontsize=12)
    plt.ylabel(r"$\mathrm{corr\_sum}$", fontsize=12)
    plt.tick_params(direction="in", length=4, width=1.0, top=True, right=True)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_corr_sum_bars(df):
    """
    Plota a média de corr_sum para cada categoria em barras laterais:
      - Antiferromagnético: 0-22 + 271-359
      - Frustrado: 23-160
      - Ferromagnético: 161-270
    """
    plt.figure(figsize=(10, 6))

    # Definir as categorias (agrupando as duas áreas verdes)
    categories = [
        ("Antiferromagnético", [(0, 22), (271, 359)], "green"),
        ("Frustrado", [(23, 160)], "orange"),
        ("Ferromagnético", [(161, 270)], "crimson")
    ]

    # Calcular a média e estatísticas para cada categoria
    category_names = []
    mean_values = []
    std_values = []
    counts = []
    colors = []
    w_ranges = []

    for label, intervals, color in categories:
        # Combinar todos os intervalos da categoria
        combined_mask = False
        w_min = float('inf')
        w_max = float('-inf')
        
        for start, end in intervals:
            mask = (df["w"] >= start) & (df["w"] <= end)
            combined_mask = combined_mask | mask
            
            # Atualizar range de w
            if mask.any():
                w_min = min(w_min, start)
                w_max = max(w_max, end)
        
        if combined_mask.any():
            mean_corr = df["corr_sum"][combined_mask].mean()
            std_corr = df["corr_sum"][combined_mask].std()
            count = combined_mask.sum()
            
            category_names.append(label)
            mean_values.append(mean_corr)
            std_values.append(std_corr)
            counts.append(count)
            colors.append(color)
            w_ranges.append(f"{w_min}-{w_max}")

    # Criar gráfico de barras
    bars = plt.bar(category_names, mean_values, color=colors, alpha=0.7, 
                   edgecolor='black', linewidth=1.2, capsize=5)

    # Adicionar barras de erro (desvio padrão)
    plt.errorbar(category_names, mean_values, yerr=std_values, fmt='none', 
                 color='black', capsize=5, capthick=1.5, elinewidth=1.5)

    # Adicionar valores nas barras
    for i, (bar, mean_val, std_val, count, w_range) in enumerate(zip(bars, mean_values, std_values, counts, w_ranges)):
        height = bar.get_height()
        
        # Valor da média
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.02, 
                f'{mean_val:.3f}', ha='center', va='bottom', fontsize=11, weight='bold')
        
        # Estatísticas abaixo da barra
        stats_text = f'n={count}\nw={w_range}'
        plt.text(bar.get_x() + bar.get_width()/2., -0.1, 
                stats_text, ha='center', va='top', fontsize=9, 
                transform=plt.gca().get_xaxis_transform())

    # Personalizar o gráfico
    plt.ylabel(r'Média de $\mathrm{corr\_sum}$', fontsize=12)
    plt.xlabel('Categoria', fontsize=12)
    plt.title('Média de corr_sum por Categoria', fontsize=14, pad=20)
    
    # Ajustar limites do eixo Y para espaço dos textos
    y_min, y_max = plt.ylim()
    plt.ylim(y_min, y_max + 0.1 * (y_max - y_min))
    
    # Estilo científico
    plt.tick_params(direction="in", length=4, width=1.0, top=True, right=True)
    plt.grid(True, linestyle="--", alpha=0.6, axis='y')
    
    plt.tight_layout()
    plt.show()

    # Mostrar tabela de resultados
    print("\n" + "="*60)
    print("RESUMO ESTATÍSTICO POR CATEGORIA")
    print("="*60)
    print(f"{'Categoria':<20} {'Média':<8} {'Std':<8} {'n':<6} {'w'}")
    print("-"*60)
    
    for i, (name, mean_val, std_val, count, w_range) in enumerate(zip(category_names, mean_values, std_values, counts, w_ranges)):
        print(f"{name:<20} {mean_val:.3f}   {std_val:.3f}   {count:<6} {w_range}")

# Versão alternativa com barras horizontais
def plot_corr_sum_horizontal_bars(df):
    """
    Plota a média de corr_sum para cada categoria em barras horizontais
    """
    plt.figure(figsize=(10, 6))

    # Definir as categorias (agrupando as duas áreas verdes)
    categories = [
        ("Antiferromagnético", [(0, 22), (271, 359)], "green"),
        ("Frustrado", [(23, 160)], "orange"),
        ("Ferromagnético", [(161, 270)], "crimson")
    ]

    # Calcular a média e estatísticas para cada categoria
    category_names = []
    mean_values = []
    std_values = []
    counts = []
    colors = []
    w_ranges = []

    for label, intervals, color in categories:
        # Combinar todos os intervalos da categoria
        combined_mask = False
        w_min = float('inf')
        w_max = float('-inf')
        
        for start, end in intervals:
            mask = (df["w"] >= start) & (df["w"] <= end)
            combined_mask = combined_mask | mask
            
            # Atualizar range de w
            if mask.any():
                w_min = min(w_min, start)
                w_max = max(w_max, end)
        
        if combined_mask.any():
            mean_corr = df["corr_sum"][combined_mask].mean()
            std_corr = df["corr_sum"][combined_mask].std()
            count = combined_mask.sum()
            
            category_names.append(label)
            mean_values.append(mean_corr)
            std_values.append(std_corr)
            counts.append(count)
            colors.append(color)
            w_ranges.append(f"{w_min}-{w_max}")

    # Criar gráfico de barras horizontais
    y_pos = np.arange(len(category_names))
    bars = plt.barh(y_pos, mean_values, color=colors, alpha=0.7, 
                    edgecolor='black', linewidth=1.2, capsize=5)

    # Adicionar barras de erro (desvio padrão)
    plt.errorbar(mean_values, y_pos, xerr=std_values, fmt='none', 
                 color='black', capsize=5, capthick=1.5, elinewidth=1.5)

    # Adicionar valores nas barras
    for i, (bar, mean_val, std_val, count, w_range) in enumerate(zip(bars, mean_values, std_values, counts, w_ranges)):
        width = bar.get_width()
        
        # Valor da média
        plt.text(width + 0.02, bar.get_y() + bar.get_height()/2., 
                f'{mean_val:.3f}', ha='left', va='center', fontsize=11, weight='bold')
        
        # Estatísticas no eixo Y
        stats_text = f'  n={count}, w={w_range}'
        plt.text(-0.1, bar.get_y() + bar.get_height()/2., 
                stats_text, ha='right', va='center', fontsize=9)

    # Personalizar o gráfico
    plt.yticks(y_pos, category_names)
    plt.xlabel(r'Média de $\mathrm{corr\_sum}$', fontsize=12)
    plt.title('Média de corr_sum por Categoria', fontsize=14, pad=20)
    
    # Ajustar limites do eixo X para espaço dos textos
    x_max = max(mean_values) + max(std_values) + 0.1
    plt.xlim(0, x_max)
    
    # Estilo científico
    plt.tick_params(direction="in", length=4, width=1.0, top=True, right=True)
    plt.grid(True, linestyle="--", alpha=0.6, axis='x')
    
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_corr_sum_mean(df):
    """
    Plota a média de corr_sum para cada categoria:
      - Antiferromagnético: 0-22 + 271-359
      - Frustrado: 23-160
      - Ferromagnético: 161-270
    """
    plt.figure(figsize=(8, 5))

    # Definir as categorias (agrupando as duas áreas verdes)
    categories = [
        ("antiferromagnético", [(0, 22), (271, 359)], "green", "o"),
        ("frustrado", [(23, 160)], "orange", "s"),
        ("ferromagnético", [(161, 270)], "crimson", "^")
    ]

    # Calcular e plotar a média para cada categoria
    w_positions = []
    mean_values = []
    colors = []
    markers = []
    labels = []

    for label, intervals, color, marker in categories:
        # Combinar todos os intervalos da categoria
        combined_mask = False
        for start, end in intervals:
            mask = (df["w"] >= start) & (df["w"] <= end)
            combined_mask = combined_mask | mask
        
        if combined_mask.any():
            mean_corr = df["corr_sum"][combined_mask].mean()
            count = combined_mask.sum()
            
            # Calcular posição w média para plotagem
            w_combined = df["w"][combined_mask].mean()
            
            w_positions.append(w_combined)
            mean_values.append(mean_corr)
            colors.append(color)
            markers.append(marker)
            labels.append(label)
            
            print(f"{label}: {mean_corr:.3f} (n={count}, w_avg={w_combined:.1f})")

    # Plotar os pontos
    for i, (w, mean_val, color, marker) in enumerate(zip(w_positions, mean_values, colors, markers)):
        plt.scatter(w, mean_val, color=color, s=120, marker=marker, zorder=5, label=labels[i])

    # Conectar os pontos com linhas
    for i in range(len(w_positions) - 1):
        plt.plot([w_positions[i], w_positions[i+1]], [mean_values[i], mean_values[i+1]], 
                linestyle="--", color="gray", alpha=0.7, linewidth=1.5)

    # Adicionar valores sobre os pontos
    for x, y in zip(w_positions, mean_values):
        plt.text(x, y + 0.02, f"{y:.3f}", ha="center", fontsize=10, color="black", weight='bold')

    # Estilo científico
    plt.xticks(range(0, 361, 30))
    plt.xlabel(r"$w$", fontsize=12)
    plt.ylabel(r"Média de $\mathrm{corr\_sum}$", fontsize=12)
    plt.tick_params(direction="in", length=4, width=1.0, top=True, right=True)
    plt.grid(True, linestyle="--", alpha=0.6)
    
    # Ajustar limites para melhor visualização
    plt.xlim(-20, 380)
    plt.ylim(min(mean_values) - 0.1, max(mean_values) + 0.1)
    
    # Legenda automática (sem repetições)
    plt.legend(frameon=False, fontsize=10, loc='best')
    
    plt.tight_layout()
    plt.show()

    # Mostrar estatísticas detalhadas
    print("\nEstatísticas detalhadas:")
    for label, intervals, color, marker in categories:
        combined_mask = False
        w_values = []
        corr_values = []
        
        for start, end in intervals:
            mask = (df["w"] >= start) & (df["w"] <= end)
            combined_mask = combined_mask | mask
            w_values.extend(df["w"][mask].tolist())
            corr_values.extend(df["corr_sum"][mask].tolist())
        
        if combined_mask.any():
            mean_corr = df["corr_sum"][combined_mask].mean()
            std_corr = df["corr_sum"][combined_mask].std()
            count = combined_mask.sum()
            w_range = f"{min(w_values)}-{max(w_values)}"
            
            print(f"  {label:20} | Média: {mean_corr:.3f} ± {std_corr:.3f} | n: {count:2d} | w: {w_range}")

In [ ]:
def plot_corr_sum_mean_4(df):
    """
    Plota a média de corr_sum para cada faixa de w:
      - 0 a 22     -> verde (antiferromagnético)
      - 23 a 160   -> laranja (frustrado)
      - 161 a 270  -> rosado (ferromagnético)
      - 271 a 359  -> verde (antiferromagnético)
    """
    plt.figure(figsize=(8, 5))

    # Definir os intervalos
    intervals = [
        (0, 22, "green", "antiferromagnético"),
        (23, 160, "orange", "frustrado"), 
        (161, 270, "crimson", "ferromagnético"),
        (271, 359, "green", "antiferromagnético")
    ]

    # Calcular e plotar a média para cada intervalo
    w_values = []
    mean_values = []
    colors = []
    labels = []

    for i, (start, end, color, label) in enumerate(intervals):
        mask = (df["w"] >= start) & (df["w"] <= end)
        if mask.any():
            mean_corr = df["corr_sum"][mask].mean()
            w_center = (start + end) / 2  # Ponto central do intervalo
            
            w_values.append(w_center)
            mean_values.append(mean_corr)
            colors.append(color)
            labels.append(label)
            
            # Plotar ponto
            plt.scatter(w_center, mean_corr, color=color, s=100, 
                       marker="o" if color == "green" else "s" if color == "orange" else "^",
                       zorder=5)

    # Plotar linhas conectando os pontos (exceto entre os intervalos desconectados)
    for i in range(len(w_values) - 1):
        # Não conectar se for mudança entre verde e verde (intervalos desconectados)
        if colors[i] == "green" and colors[i+1] == "green":
            continue
        plt.plot([w_values[i], w_values[i+1]], [mean_values[i], mean_values[i+1]], 
                linestyle="--", color="gray", alpha=0.7)

    # Adicionar valores sobre os pontos
    for x, y in zip(w_values, mean_values):
        plt.text(x, y + 0.02, f"{y:.2f}", ha="center", fontsize=9, color="black", weight='bold')

    # Criar legenda personalizada sem repetições
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='green', linestyle='-', 
               label='antiferromagnético', markersize=8),
        Line2D([0], [0], marker='s', color='orange', linestyle='-', 
               label='frustrado', markersize=8),
        Line2D([0], [0], marker='^', color='crimson', linestyle='-', 
               label='ferromagnético', markersize=8)
    ]

    plt.legend(handles=legend_elements, frameon=False, fontsize=10)

    # Estilo científico
    plt.xticks(range(0, 361, 30))
    plt.xlabel(r"$w$", fontsize=12)
    plt.ylabel(r"Média de $\mathrm{corr\_sum}$", fontsize=12)
    plt.tick_params(direction="in", length=4, width=1.0, top=True, right=True)
    plt.grid(True, linestyle="--", alpha=0.6)
    
    # Ajustar limites para melhor visualização
    plt.xlim(-10, 370)
    
    plt.tight_layout()
    plt.show()

    # Opcional: Mostrar os valores calculados
    print("Médias por intervalo:")
    for i, (start, end, color, label) in enumerate(intervals):
        mask = (df["w"] >= start) & (df["w"] <= end)
        if mask.any():
            mean_corr = df["corr_sum"][mask].mean()
            count = mask.sum()
            print(f"  {start}-{end} ({label}): {mean_corr:.3f} (n={count})")

In [ ]:
def plot(L,IT,str_nets,wT,l_info,exact_gs_energy,
         iters_Jastrow_ds,energy_Jastrow_ds,iters_Net_ds,energy_Net_ds, show):
    wfig, ax = plt.subplots(1, 1, figsize=(8, 6), sharex=True); x_pos = 0.4; y_pos = 0.94
    w = int(wT);  
    f_wt = ftheta(w)
    l_tp = []; l_tp.append(f_wt)
    # ========== Plotting ==========
    # Plot RBM
    ax.plot(iters_Net_ds, energy_Net_ds, label=f'{l_info[0]}({energy_Net_ds[-1]:.3f})', 
            linestyle=linestyles[0], marker=markers[0], 
            color=colors[0], markersize=5)

    # Plot Jastrow
    ax.plot(iters_Jastrow_ds, energy_Jastrow_ds, label=f'JASTROW ({energy_Jastrow_ds[-1]:.3f})', 
            linestyle=linestyles[1], marker=markers[1], 
            color=colors[1], markersize=5)
    if exact_gs_energy != 0:
        ax.axhline(y=exact_gs_energy, color='k', lw=2, ls='--', label=f'Exact ({exact_gs_energy:.3f})')

    # Axis configurations
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_minor_locator(FixedLocator([]))
    
    ax.set_ylabel("Energia")
    ax.set_xlabel("Interações")
    
    ax.text(x_pos, y_pos, f'(a) $L={L}$ | {l_tp[0]}',transform=ax.transAxes, fontsize=12, verticalalignment='top')
    ax.legend(fontsize=9, loc='upper right', frameon=False)


    # Save and show
    pathfg = f"fig/hhh/EGS_L_{L}_IT_{IT}_W_{wT}_{str_nets}.png"
    print(pathfg)
    plt.savefig(pathfg, dpi=300, bbox_inches='tight')
    if show == 1:
        plt.show()
    plt.close()

In [ ]:
def plot_w_i(ex_n,df_w,L, IT,str_nets,kernel_bias,real_imag,qtdeN, show):
    wfig, ax = plt.subplots(1, 1, figsize=(8, 6), sharex=True); x_pos = 0.4; y_pos = 0.94

    x1        = df_w["id"]
    stride    = 5; index = 0; ncol1 = len(df_w) - 2

    if ncol1 > qtdeN:
        limit_col = qtdeN
    else:
        limit_col = ncol1
    
    for i in range(0,qtdeN):
        nn = random.randint(1, ncol1) 
        w1 = df_w[str(nn)]
        x1_ds, w1_ds = downsample(x1, w1, stride) 
        index = i + 1
        label = r"w" + str(index)
        ax.plot(x1_ds, w1_ds, label=label, linestyle=linestyles[i], marker=markers[i], color=colors[i], markersize=5)
    
    ax.tick_params(direction='in', length=4, width=1.0, top=True, bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, bottom=True, left=True, right=True)
    ax.tick_params(labelbottom=False)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_minor_locator(FixedLocator([]))
    ax.set_xlabel(r"Iteractions")
    ax.set_ylabel(r"Weights")
    plt.subplots_adjust(hspace=0.1, wspace=0.1)

    # Save and show
    path = 'fig/fff/'
    create(path)
    pathfg = f"{path}WG_IT_L_{L}_IT_{IT}_W_{wT}_QTN_{qtdeN}_{str_nets}.png"
    
    print(pathfg)
    plt.savefig(pathfg, dpi=300, bbox_inches='tight')
    if show == 1:
        plt.show()
    plt.close()

In [ ]:
def plot_m_w_i(ex_n,df_w,L, IT,str_nets,kernel_bias,real_imag,qtdeN, opc, show):
    wfig, ax = plt.subplots(1, 1, figsize=(8, 6), sharex=True); x_pos = 0.4; y_pos = 0.94

    i = 0
    x1        = df_w["id"]
    stride    = 10; index = 0; ncol1 = len(df_w) - 2
   
    label = r"median (w)"
    median1 = df_w.iloc[:, 1:].median(axis=1).round(4)
    i  = i + 1 
    x1_ds, median1_ds = downsample(x1, median1, stride)
    if (opc == 1) or (opc == 0) :
        ax.plot(x1_ds, median1_ds, label=label, linestyle=linestyles[i],marker=markers[i], color=colors[i], markersize=5,alpha=0.5)
        ax.set_ylabel(r"Metrics Weights")

    label = r"std (w)"
    std1 = df_w.iloc[:, 1:].std(axis=1).round(5)
    i  = i + 1 
    x1_ds, std1_ds = downsample(x1, std1, stride)
    if (opc == 2) or (opc == 0):
        ax.plot(x1_ds, std1_ds, label=label, linestyle=linestyles[i],marker=markers[i], color=colors[i], markersize=5,alpha=0.5)
        ax.set_ylabel(r"Metrics Weights")

    label = r"avg (w)"
    if (opc == 3) or (opc == 0) :
        if opc == 3: 
            m1 = df_w.iloc[:, 1:].mean(axis=1)
            ax.ticklabel_format(axis='y', style='plain', useOffset=False)
        else:
            m1 = df_w.iloc[:, 1:].mean(axis=1).round(10)
        i  = i + 1 
        x1_ds, m1_ds = downsample(x1, m1, stride)
        ax.plot(x1_ds, m1_ds , label=label, linestyle=linestyles[i],marker=markers[i], color=colors[i], markersize=5,alpha=0.5)
        ax.set_ylabel(r"Metrics Weights")

    
    label = r"log(amp) (w)"
    range1 = (df_w.iloc[:, 1:].max(axis=1) - df_w.iloc[:, 1:].min(axis=1)).round(3)
    range1 = np.log10(range1)  
    i  = i + 1 
    x1_ds, range1_ds = downsample(x1, range1, stride)
    if (opc == 4) or (opc == 0) :
        ax.plot(x1_ds, range1_ds, label=label, linestyle=linestyles[i],marker=markers[i], color=colors[i], markersize=5,alpha=0.5)
        ax.set_ylabel(r"Metrics Weights")

    label = r"log(disp) (w)"
    cv1 = (df_w.iloc[:, 1:].std(axis=1) / df_w.iloc[:, 1:].mean(axis=1)).round(3)
    cv1 = np.log10(cv1)        # Log base 10

    i  = i + 1 
    x1_ds, cv1_ds = downsample(x1, cv1, stride)
    if (opc == 5) or (opc == 0):
        ax.plot(x1_ds, cv1_ds, label=label, linestyle=linestyles[i],marker=markers[i], color=colors[i], markersize=5,alpha=0.5)
        ax.set_ylabel(r"Metrics Weights")

   # all_y_values = np.concatenate([
   #      m1_ds, m1_ds + std1_ds, m1_ds - std1_ds,
   #     median1_ds, range1_ds, cv1_ds  # Adicione aqui
   # ])


    #y_padding = 0.1 * (np.nanmax(all_y_values) - np.nanmin(all_y_values))
    #y_min = np.nanmin(all_y_values) - y_padding
    #y_max = np.nanmax(all_y_values) + y_padding

    #ax.set_ylim(y_min, y_max)
    # Configurações do eixo
    ax.tick_params(direction='in', length=4, width=1.0, top=True, 
                   bottom=True, left=True, right=True)
    ax.tick_params(which='minor', direction='in', length=2, 
                   width=0.8, top=True, bottom=True, left=True, right=True)
    ax.tick_params(labelbottom=False)

    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax.yaxis.set_minor_locator(FixedLocator([]))

    ax.set_xlabel(r"Iteractions")
    
    # Mover a legenda para fora à esquerda
    ax.legend(fontsize=9, loc='upper right', bbox_to_anchor=(1.28, 1), 
              frameon=True, framealpha = 1, edgecolor = 'black')
    
    # Save and show
    path = 'fig/www/'
    create(path)
    pathfg = f"{path}M_WG_IT_L_{L}_IT_{IT}_W_{wT}_QTN_{str_nets}.png"
    
    print(pathfg)
    plt.savefig(pathfg, dpi=300, bbox_inches='tight')
    if show == 1:
        plt.show()
    plt.close()